In [0]:
# To create Text widget to get source file path
# dbutils.widgets.text("Source Folder Path", "")

In [0]:
%pip install databricks-feature-engineering

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.feature_store import FeatureStoreClient

In [0]:
source_folder_path = dbutils.widgets.get("Source Folder Path")
if not source_folder_path:
    dbutils.notebook.exit("source_folder_path is not provided.")

In [0]:
expected_files = ["customer_features.csv", "product_features.csv", "training_labels.csv", "inference_data.csv"]

# Check if the expected files exist in the source folder path
existing_files = dbutils.fs.ls(source_folder_path)
existing_files_list = [file.name for file in existing_files]

missing_files = [file for file in expected_files if file not in existing_files_list]

if missing_files:
    dbutils.notebook.exit(f"Missing files: {', '.join(missing_files)}")

In [0]:
db = "sales"
query = f"CREATE DATABASE IF NOT EXISTS {db}"
spark.sql(query)

In [0]:
# Configuration

feature_tables = {"customer_features": {"primary_keys": ["customer_id"], "file_name":"customer_features.csv"}, 
                  "product_features": {"primary_keys": ["product_id"], "file_name":"product_features.csv"}}

In [0]:
fs = FeatureStoreClient()

# Create Feature Tables

for table in feature_tables:
    table_name = f"{db}.{table}"
    # Drop table if it already exists
    try:
        fs.drop_table(name=table_name)
        print(f"Dropped existing table: {table_name}")
    except Exception as e:
        print(f"No existing table to drop or error occurred: {e}")

    feature_file = f"{source_folder_path}/{feature_tables.get(table).get('file_name')}"
    df = spark.read.load(feature_file, format="csv",sep=",",inferSchema="true",header="true" )


    # Create table
    fs.create_table(
        name=table_name,
        primary_keys=feature_tables.get(table).get("primary_keys"),
        df=df,
        schema=df.schema,
        description="Customer feature table for wine preferences",
    )

    print("Feature table recreated successfully.")


